In [83]:
!pip install xgboost lightgbm

In [84]:
# ==============================================================================
# 1. MANIPULACIÓN DE DATOS Y ESTRUCTURAS MATEMÁTICAS
# ==============================================================================
# Pandas es la librería reina para manejar "DataFrames" (tablas de datos con filas y columnas).
import pandas as pd

# Numpy se usa para operaciones matemáticas avanzadas y manejo de vectores o matrices (arrays).
import numpy as np


# ==============================================================================
# 2. VALIDACIÓN Y OPTIMIZACIÓN DE MODELOS
# ==============================================================================
# train_test_split: Sirve para dividir tus datos en dos grupos: uno para entrenar el modelo (train) y otro para ponerlo a prueba (test).
# GridSearchCV: Es una herramienta que prueba automáticamente muchas combinaciones de "hiperparámetros" (configuraciones del modelo) para encontrar la mejor.
from sklearn.model_selection import train_test_split, GridSearchCV


# ==============================================================================
# 3. AUTOMATIZACIÓN DE PROCESOS (PIPELINE)
# ==============================================================================
# Pipeline: Permite encadenar pasos secuenciales (como limpiar datos, normalizarlos y luego entrenar el modelo) en un solo objeto, evitando fugas de datos (data leakage).
from sklearn.pipeline import Pipeline


# ==============================================================================
# 4. MODELOS DE MACHINE LEARNING (Clasificadores)
# ==============================================================================
# Nota: Todos estos modelos sirven para predecir categorías (ej. "Es fraude / No es fraude", "Cliente VIP / Regular").

# Regresión Logística: A pesar de su nombre, es un modelo lineal clásico para clasificación binaria.
from sklearn.linear_model import LogisticRegression

# Support Vector Machines (Máquinas de Vectores de Soporte): Clasificador poderoso que busca maximizar el margen de separación entre clases.
from sklearn.svm import SVC

# Árbol de Decisión: Un modelo basado en reglas de "sí/no" (como un diagrama de flujo) para clasificar los datos.
from sklearn.tree import DecisionTreeClassifier  #  Correcto

# K-Nearest Neighbors (K-Vecinos más Cercanos): Clasifica un dato según a qué grupo pertenecen los datos más cercanos a él.
from sklearn.neighbors import KNeighborsClassifier

# --- Modelos de Ensamble (Combinan muchos árboles de decisión para ser más precisos) ---

# Random Forest: Crea un "bosque" de árboles de decisión independientes y hace que voten para dar el resultado final.
# Gradient Boosting: Entrena árboles de forma secuencial, donde cada nuevo árbol corrige los errores del anterior.
# AdaBoost: Similar a Gradient Boosting, pero ajusta los pesos de los datos en cada paso para enfocarse en los más difíciles de clasificar.
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier

# XGBoost: Una versión ultra-optimizada y rapidísima de Gradient Boosting. Muy usada en competencias de Ciencia de Datos (como Kaggle).
from xgboost import XGBClassifier

# LightGBM: Otra variante de Gradient Boosting desarrollada por Microsoft, diseñada para ser extremadamente rápida y manejar grandes volúmenes de datos.
from lightgbm import LGBMClassifier

# --- Modelos Basados en Probabilidad (Naive Bayes) ---

# GaussianNB: Modelo basado en el Teorema de Bayes, asume que las variables continuas siguen una distribución normal (campana de Gauss).
# BernoulliNB: Variante de Naive Bayes ideal para datos que son binarios (ceros y unos, o respuestas de sí/no).
from sklearn.naive_bayes import GaussianNB, BernoulliNB


# ==============================================================================
# 5. MÉTRICAS DE EVALUACIÓN
# ==============================================================================
# accuracy_score: Mide el porcentaje de predicciones correctas que hizo el modelo (Aciertos totales / Total de casos).
from sklearn.metrics import accuracy_score


# ==============================================================================
# 6. PERSISTENCIA DE MODELOS (Guardado)
# ==============================================================================
# Pickle: Es una librería nativa de Python que sirve para "serializar" objetos. En ML la usamos para guardar el modelo ya entrenado en un archivo (.pkl) y poder usarlo después en producción sin tener que volver a entrenarlo.
import pickle

In [85]:
df = pd.read_csv('../data/titanic_procesado.csv')

In [86]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


In [87]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [88]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42) # Dividir los datos en entrenamiento (80%) y prueba (20%) y random_state para reproducibilidad
X_train = X_train.values  # Convertir a NumPy array
y_train = y_train.values  # Convertir a NumPy array
X_test = X_test.values    # Convertir a NumPy array
y_test = y_test.values    # Convertir a NumPy array #la función train_test_split mezcla los datos de forma completamente aleatoria antes de dividirlos. Si corres el código hoy, 
#se dividirá de una forma; si lo corres mañana, se dividirá de otra. 
#Esto haría que las métricas de tu modelo (como el accuracy) cambien cada vez que ejecutes el notebook. por eso se coloca un numero fijo en random_state, para que la división de los datos sea siempre la misma cada vez que ejecutes el código. Esto es crucial para poder comparar resultados de manera consistente.

In [89]:
# Definir un diccionario principal que agrupa los nombres de los algoritmos como claves
modelos = {
    # --- Configuración para la Regresión Logística ---
    'Regresión Logística': {
        'modelo': LogisticRegression(), # Instancia el objeto del modelo lineal básico
        'parametros': { # Diccionario con las perillas que GridSearch va a combinar y probar
            'C': [0.01, 0.1, 1, 10, 100], # Valores de regularización (menor valor = penalización más fuerte)
            'penalty': ['l1', 'l2'], # Tipo de regularización: L1 (Lasso, borra variables) o L2 (Ridge, reduce pesos)
            'solver': ['liblinear', 'saga'], # Algoritmos matemáticos de optimización para resolver la ecuación
            'max_iter': [100, 500, 1000] # Límite máximo de iteraciones para que el algoritmo converja
        }
    },
    # --- Configuración para la Máquina de Vectores de Soporte ---
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(), # Instancia el modelo geométrico de soporte de vectores
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'], # Tipo de plano divisor (recto, polinomial, curvo)
            'C': [0.1, 1, 10] # Nivel de tolerancia a errores de clasificación cerca del margen
        }
    },
    # --- Configuración para el Árbol de Decisión Único ---
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(), # Instancia el árbol de decisiones básico
        'parametros': {
            'splitter': ['best', 'random'], # Estrategia para elegir la división en cada nodo (mejor o al azar)
            'max_depth': [None, 1, 2, 3, 4] # Niveles máximos del árbol (None significa expandir hasta que sea puro)
        }
    },
    # --- Configuración para Random Forest (Ensamble en Paralelo) ---
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(), # Instancia el conjunto de árboles independientes
        'parametros': {
            'n_estimators': [10, 100], # Número de árboles a construir en el bosque
            'max_depth': [None, 1, 2, 3, 4], # Profundidad límite de los árboles del bosque
            'max_features': ['sqrt', 'log2', None] # Cantidad de columnas que cada árbol puede evaluar por nodo
        }
    },
    # --- Configuración para Gradient Boosting (Ensamble Secuencial Clásico) ---
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(), # Instancia el ensamble corrector de errores paso a paso
        'parametros': {
            'n_estimators': [10, 100], # Cantidad de árboles secuenciales que se van a encadenar
            'max_depth': [None, 1, 2, 3, 4] # Profundidad asignada a cada árbol individual de la cadena
        }
    },
    # --- Configuración para AdaBoost ---
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(), # Instancia el clasificador basado en pesos adaptativos
        'parametros': {
            'n_estimators': [10, 100] # Cantidad de modelos débiles que se van a acoplar consecutivamente
        }
    },
    # --- Configuración para KNN (Basado en Instancias/Vecinos) ---
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(), # Instancia el buscador de vecindad de datos
        'parametros': {
            'n_neighbors': [3, 5, 7] # Cuántos vecinos más cercanos se consultan para definir el grupo
        }
    },
    # --- Configuración para XGBoost (Boosting Optimizado de Alto Rendimiento) ---
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(), # Instancia el algoritmo Extreme Gradient Boosting
        'parametros': {
            'n_estimators': [10, 100], # Total de árboles secuenciales de corrección fina
            'max_depth': [None, 1, 2, 3] # Control estricto de profundidad por árbol para evitar overfitting
        }
    },
    # --- Configuración para LightGBM (Boosting Veloz de Microsoft) ---
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(), # Instancia el modelo basado en histogramas eficientes
        'parametros': {
            'n_estimators': [10, 100], # Número de árboles en su secuencia acelerada
            'max_depth': [None, 1, 2, 3], # Profundidad límite de los nodos del árbol
            'learning_rate': [0.1, 0.2, 0.3], # Factor de escala que controla el tamaño de los pasos de aprendizaje
            'verbose': [-1] # Apaga alertas innecesarias en la consola para limpiar la salida de texto
        }
    },
    # --- Configuración para Gaussian Naive Bayes ---
    'GaussianNB': {
        'modelo': GaussianNB(), # Instancia el modelo probabilístico para datos continuos
        'parametros': {} # Se deja vacío porque este algoritmo matemático puro no requiere hiperparámetros
    },
    # --- Configuración para Bernoulli Naive Bayes ---
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(), # Instancia el modelo probabilístico para variables binarias
        'parametros': {
            'alpha': [0.1, 1.0, 10.0] # Parámetro de suavizado para evitar probabilidades de cero absoluto
        }
    }
}

# Crear una lista vacía para ir guardando el diccionario de rendimiento de cada modelo
puntajes_modelos = []
# Declarar una variable de control numérica con el peor resultado posible para empezar a comparar
mejor_precision = 0
# Inicializar como vacía la variable que almacenará el objeto del mejor modelo tuneado
mejor_estimador = None
# Inicializar como vacía la variable de texto para el nombre del modelo ganador
mejor_modelo = None
# Crear un diccionario vacío para guardar las mejores versiones entrenadas de cada algoritmo
estimadores = {}

# Iniciar un bucle 'for' que desempaca el nombre del modelo y su bloque de información interna
for nombre, info_modelo in modelos.items():
    # Configurar el buscador de rejilla (GridSearchCV) para el algoritmo en turno
    grid_search = GridSearchCV(
        estimator=info_modelo['modelo'], # El algoritmo base que se va a procesar en esta iteración
        param_grid=info_modelo['parametros'], # El set de combinaciones de hiperparámetros declarados arriba
        cv=5, # Validación cruzada de 5 pliegues (entrena 5 veces variando los datos para evitar sesgos)
        scoring='accuracy', # Métrica objetivo para decidir cuál combinación es la mejor (precisión global)
        verbose=0, # Nivel de detalle en logs (0 desactiva los mensajes automáticos de progreso)
        n_jobs=-1, # Usa el 100% de los núcleos del procesador de tu computadora para trabajar en paralelo
    )

    # Disparar el proceso masivo de entrenamiento y validación cruzada con los datos de práctica
    grid_search.fit(X_train, y_train)
    
    # Evaluar el mejor modelo resultante usando el set de prueba X_test que estaba reservado
    y_pred = grid_search.predict(X_test)
    
    # Calcular numéricamente el porcentaje de aciertos contrastando las predicciones contra las etiquetas reales
    precision = accuracy_score(y_test, y_pred)
    
    # Insertar un nuevo diccionario en la lista con los resultados obtenidos en este ciclo
    puntajes_modelos.append({
        'Modelo': nombre, # Nombre comercial/amigable del algoritmo
        'Precisión': precision # El score matemático que obtuvo en test
    })

    # Guardar en nuestro diccionario de estimadores la configuración ganadora de este modelo específico
    estimadores[nombre] = grid_search.best_estimator_
    
    # Validar mediante una condicional si el rendimiento actual supera al récord histórico guardado
    if precision > mejor_precision:
        mejor_modelo = nombre # Sobrescribir el nombre del líder de la tabla
        mejor_precision = precision # Actualizar la marca a vencer con el nuevo porcentaje más alto
        mejor_estimador = grid_search.best_estimator_ # Guardar el objeto matemático campeón para usarlo después

# Transformar la lista de diccionarios de resultados en una tabla estructurada de Pandas y ordenarla de mayor a menor
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir un encabezado decorativo en la consola del notebook
print("Rendimiento de los modelos de clasificación")
# Desplegar la tabla ordenada redondeando los decimales a solo 2 dígitos para facilitar la lectura
print(metricas.round(2))

# Imprimir un separador visual en texto
print('---------------------------------------------------')
# Mostrar la sección de conclusiones finales
print("MEJOR MODELO DE CLASIFICACIÓN")
# Imprimir mediante un string formateado (f-string) el nombre del algoritmo que ganó la competencia
print(f"Modelo: {mejor_modelo}")
# Imprimir la precisión final del campeón formateada para mostrar únicamente 2 decimales
print(f"Precisión: {mejor_precision:.2f}")

c:\Users\direc\OneDrive\Documentos\4. Hydbridge Ingeniería en SoftWare\IA_4to\aprendizaje_supervisado_Titanic\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
4     Clasificador de Gradient Boosting       0.83
6      Clasificador K-Nearest Neighbors       0.83
7                  Clasificador XGBoost       0.82
8                     Clasificador LGBM       0.81
1   Clasificador de Vectores de Soporte       0.80
2     Clasificador de Árbol de Decisión       0.80
0                   Regresión Logística       0.79
5                 Clasificador AdaBoost       0.79
3    Clasificador de Bosques Aleatorios       0.79
10             Clasificador Naive Bayes       0.79
9                            GaussianNB       0.76
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador de Gradient Boosting
Precisión: 0.83


c:\Users\direc\OneDrive\Documentos\4. Hydbridge Ingeniería en SoftWare\IA_4to\aprendizaje_supervisado_Titanic\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [90]:
# Esto ya lo tenemos importado. Lo ponemos nuevamente nada más de referencia
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


# Creamos el modelo de regresión logística
model = LogisticRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

# Realizamos predicciones con el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluamos el modelo usando precisión
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")

model = LogisticRegression(
    C=0.5,                  # Valor de regularización
    penalty='l2',            # Tipo de penalización (l2 es la regularización Ridge)
    solver='lbfgs',         # Algoritmo de optimización
    max_iter=200,           # Número máximo de iteraciones
    class_weight='balanced' # Ajustar pesos de las clases
)

# Entrenar el modelo
model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar la precisión del modelo
accuracy = accuracy_score(y_test, y_pred)

print(f"Precisión del modelo: {accuracy:.2f}")




Precisión del modelo: 0.80
Precisión del modelo: 0.78


c:\Users\direc\OneDrive\Documentos\4. Hydbridge Ingeniería en SoftWare\IA_4to\aprendizaje_supervisado_Titanic\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [91]:
# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

In [92]:
# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    grid_search = GridSearchCV(
    estimator=info_modelo['modelo'],
    param_grid=info_modelo['parametros'],
    cv=5,
    scoring='accuracy',
    verbose=0,
    n_jobs=-1,
)
    
# Ajustar GridSearchCV con los datos de entrenamiento
grid_search.fit(X_train, y_train)

# Hacer predicciones con el modelo ajustado
y_pred = grid_search.predict(X_test)

# Calcular la precisión de las predicciones
precision = accuracy_score(y_test, y_pred)

# Almacenar los resultados del modelo
puntajes_modelos.append({
    'Modelo': nombre,
    'Precisión': precision
})

estimadores[nombre] = grid_search.best_estimator_

# Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
if precision > mejor_precision:
    mejor_modelo = nombre
    mejor_precision = precision
    mejor_estimador = grid_search.best_estimator_


# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

Rendimiento de los modelos de clasificación
                     Modelo  Precisión
0  Clasificador Naive Bayes       0.79
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador Naive Bayes
Precisión: 0.79


In [94]:
import pickle

In [95]:
with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)  